In [43]:
from pathlib import Path
from typing import Optional, Any
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd, numpy as np
import os

In [34]:
df_iot:pd.DataFrame = pd.DataFrame(pd.read_csv(r"../datas/gold/mqtt_iot_clean.csv"))
df_plc:pd.DataFrame = pd.DataFrame(pd.read_csv(r"../datas/gold/mqtt_plc_clean.csv"))
df_iot = (
    df_iot.sort_values("timestamp")
      .reset_index(drop=True)
)
df_plc = (
    df_plc.sort_values("timestamp")
      .reset_index(drop=True)
)

In [35]:
df_iot.head(45)

,timestamp,machine_id,iot_vitesse_rotation,iot_courant_moteur,iot_pression_hydraulique,iot_temperature,iot_vibration_peak,iot_charge_moteur
0,2026-06-01 00:00:00,MCH-001,6000.0,19.815780,0.000000,48.711109,1.267143,45.0
1,2026-06-01 00:00:00,MCH-002,4500.0,22.085784,0.000000,40.794459,1.257802,45.0
2,2026-06-01 00:00:00,MCH-007,4800.0,16.279980,0.258873,39.129279,0.508685,45.0
3,2026-06-01 00:00:00,MCH-004,1250.0,43.193466,0.474612,56.080133,1.086669,80.0
4,2026-06-01 00:00:00,MCH-006,0.0,54.966129,180.021158,61.644770,3.206393,70.0
5,2026-06-01 00:00:00,MCH-005,4300.0,24.147071,0.000000,43.365719,1.445014,45.0
6,2026-06-01 00:00:00,MCH-003,5000.0,14.950068,0.000000,40.556213,0.663997,45.0
7,2026-06-01 00:00:30,MCH-005,4300.0,24.030777,0.000000,43.487068,1.399008,45.0
8,2026-06-01 00:00:30,MCH-001,6000.0,20.028414,0.000000,48.552522,1.355303,45.0
9,2026-06-01 00:00:30,MCH-004,1250.0,43.183121,0.545420,56.214722,1.028371,80.0


In [48]:
def get_dynamic_max_workers(group_count: int) -> int:
    cpu_count = os.cpu_count() or 1

    return min(
        group_count,
        max(1, cpu_count - 1)
    )
def verify_columns(df: pd.DataFrame, columns: list[str])->None:
    missing_columns:list[str] = []
    for column in columns:
        if column not in df.columns:
            missing_columns.append(column)
    if len(missing_columns) != 0:
        raise ValueError(f"Missing column(s) in dataset: \"{", ".join(missing_columns)}\"")
def verify_all_line_have_same_timestamp(df: pd.DataFrame)->None:
    timestamps:list[str] = df["timestamp"].unique().tolist()
    if len(timestamps) != 1:
        raise ValueError("All lines must have the same timestamp")

class Sensor:
    def __init__(self, timestamp: str, id_machine: str, others_data: dict[str, Any]):
        self.timestamp:str = timestamp
        self.id_machine:str = id_machine
        self.others_data:dict[str, Any] = others_data
    def get_data(self) -> dict[str, Any]:
        return {
            "timestamp": self.timestamp,
            "id_machine": self.id_machine,
            **self.others_data
        }

def create_list_of_list_of_sensor(df: pd.DataFrame, columns: list[str])-> list[Sensor]:
    """

    :param df: DataFrame ou toutes les lignes ont le meme timestamp
    :param columns: colonnes du DataFrame
    :return: liste de liste de capteurs : tous seront à envoyer simultanément au broker MQTT
    """
    verify_columns(df,columns)
    verify_all_line_have_same_timestamp(df)
    list_of_machines_sensors:list[Sensor] = []
    for line in df.itertuples(): # iteration sur chaque ligne (une ligne par machine)
        list_of_machines_sensors.append(
            Sensor(
                timestamp=line.timestamp,
                id_machine=line.machine_id,
                others_data={"iot_vitesse_rotation": line.iot_vitesse_rotation}
            )
        )
        list_of_machines_sensors.append(
            Sensor(
                timestamp=line.timestamp,
                id_machine=line.machine_id,
                others_data={"iot_courant_moteur": line.iot_vitesse_rotation}
            )
        )
        list_of_machines_sensors.append(
            Sensor(
                timestamp=line.timestamp,
                id_machine=line.machine_id,
                others_data={"iot_pression_hydraulique": line.iot_vitesse_rotation}
            )
        )
        list_of_machines_sensors.append(
            Sensor(
                timestamp=line.timestamp,
                id_machine=line.machine_id,
                others_data={"iot_temperature": line.iot_vitesse_rotation}
            )
        )
        list_of_machines_sensors.append(
            Sensor(
                timestamp=line.timestamp,
                id_machine=line.machine_id,
                others_data={"iot_vibration_peak": line.iot_vitesse_rotation}
            )
        )
        list_of_machines_sensors.append(
            Sensor(
                timestamp=line.timestamp,
                id_machine=line.machine_id,
                others_data={"iot_charge_moteur": line.iot_vitesse_rotation}
            )
        )

    return list_of_machines_sensors

create_list_of_list_of_sensor(df=df_iot.head(7), columns=["timestamp", "machine_id", "iot_vitesse_rotation", "iot_courant_moteur", "iot_pression_hydraulique", "iot_temperature", "iot_vibration_peak", "iot_charge_moteur"])

In [54]:
def split_df_by_timestamp_and_create_list_of_sensor(df: pd.DataFrame)->list[list[Sensor]]:
    columns = ["timestamp", "machine_id", "iot_vitesse_rotation", "iot_courant_moteur", "iot_pression_hydraulique", "iot_temperature", "iot_vibration_peak", "iot_charge_moteur"]
    verify_columns(df,columns)
    results:list[list[Sensor]] = []
    with ThreadPoolExecutor(max_workers=get_dynamic_max_workers(60)) as executor:
        futures = []

        for group_value, group_df in df.groupby("timestamp"):
            future = executor.submit(
                create_list_of_list_of_sensor,
                group_df,
                columns
            )
            futures.append(future)

        for future in as_completed(futures):
            results.append(future.result())
    return results

In [55]:
list_of_list_of_sensor:list[list[Sensor]] = split_df_by_timestamp_and_create_list_of_sensor(df_iot)

[<__main__.Sensor object at 0x787674be2a50>, <__main__.Sensor object at 0x787674be39b0>, <__main__.Sensor object at 0x787674be3a10>, <__main__.Sensor object at 0x787674bef140>, <__main__.Sensor object at 0x787674bef080>, <__main__.Sensor object at 0x787674beefc0>, <__main__.Sensor object at 0x787674bf85c0>, <__main__.Sensor object at 0x787674bf9580>, <__main__.Sensor object at 0x787674bf8470>, <__main__.Sensor object at 0x787674bf8410>, <__main__.Sensor object at 0x787674bf84a0>, <__main__.Sensor object at 0x787674bf82f0>, <__main__.Sensor object at 0x787674bf83e0>, <__main__.Sensor object at 0x787674bf8500>, <__main__.Sensor object at 0x787674bf8230>, <__main__.Sensor object at 0x787674bf8380>, <__main__.Sensor object at 0x787674bf83b0>, <__main__.Sensor object at 0x787674bf8320>, <__main__.Sensor object at 0x787674fffd70>, <__main__.Sensor object at 0x787674fffe90>, <__main__.Sensor object at 0x787674fffe60>, <__main__.Sensor object at 0x787674fecf20>, <__main__.Sensor object at 0x78